In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

def get_puf_embedding(X):
    """
    Computes the standard Arbiter PUF feature map efficiently.
    For a challenge vector x, phi_j = prod_{k=j}^n (1 - 2*x_k)
    """
    V = 1 - 2 * X
    return np.cumprod(V[:, ::-1], axis=1)[:, ::-1]

def my_latent(X, y):
    """
    Alternating optimization with a uniform prior over latent variables.
    Returns: w (17,), b (float), z (8000,)
    """
    N, D = X.shape # N=8000, D=16
    
    # Randomly initialize latent variables to break symmetry
    z = np.random.randint(0, 2, size=N)
    
    clf = LogisticRegression(C=2.0, max_iter=200, tol=1e-4, solver='lbfgs')
    
    for iteration in range(10):
        # 1. Reconstruct 17-bit challenges using current z estimations
        X_17 = np.zeros((N, D + 1))
        X_17[:, :8] = X[:, :8]
        X_17[:, 8] = z
        X_17[:, 9:] = X[:, 8:]
        
        # Compute embeddings and fit the 17-bit model
        Phi_17 = get_puf_embedding(X_17)
        clf.fit(Phi_17, y)
        
        w = clf.coef_[0]
        b = clf.intercept_[0]
        
        # 2. Update latent variables z by evaluating both possible choices
        X_z0 = np.zeros((N, D + 1))
        X_z0[:, :8] = X[:, :8]
        X_z0[:, 8] = 0
        X_z0[:, 9:] = X[:, 8:]
        Phi_z0 = get_puf_embedding(X_z0)
        margin_z0 = (2 * y - 1) * (np.dot(Phi_z0, w) + b)
        
        X_z1 = np.zeros((N, D + 1))
        X_z1[:, :8] = X[:, :8]
        X_z1[:, 8] = 1
        X_z1[:, 9:] = X[:, 8:]
        Phi_z1 = get_puf_embedding(X_z1)
        margin_z1 = (2 * y - 1) * (np.dot(Phi_z1, w) + b)
        
        z_new = np.where(margin_z1 > margin_z0, 1, 0)
        
        if np.array_equal(z, z_new):
            break
        z = z_new

    return w, b, z.astype(bool)

def my_latent_updated(X, y):
    """
    Alternating optimization where latent variables are regularized by a 16-bit PUF model.
    Returns: w (17,), b (float), u (16,), a (float)
    """
    N, D = X.shape
    
    # Initialize latent variables randomly
    z = np.random.randint(0, 2, size=N)
    
    clf_17 = LogisticRegression(C=2.0, max_iter=200, tol=1e-4, solver='lbfgs')
    clf_16 = LogisticRegression(C=2.0, max_iter=200, tol=1e-4, solver='lbfgs')
    
    Phi_16 = get_puf_embedding(X)
    
    for iteration in range(12):
        # 1. Update the 17-bit model using current z estimates
        X_17 = np.zeros((N, D + 1))
        X_17[:, :8] = X[:, :8]
        X_17[:, 8] = z
        X_17[:, 9:] = X[:, 8:]
        Phi_17 = get_puf_embedding(X_17)
        
        clf_17.fit(Phi_17, y)
        w = clf_17.coef_[0]
        b = clf_17.intercept_[0]
        
        # 2. Update the 16-bit model using current z targets
        clf_16.fit(Phi_16, z)
        u = clf_16.coef_[0]
        a = clf_16.intercept_[0]
        
        # 3. Jointly update latent variables z evaluating the structural prior
        X_z0 = np.zeros((N, D + 1))
        X_z0[:, :8] = X[:, :8]
        X_z0[:, 8] = 0
        X_z0[:, 9:] = X[:, 8:]
        Phi_z0 = get_puf_embedding(X_z0)
        
        # Log-likelihood components for z = 0
        log_p_r_z0 = -np.log(1.0 + np.exp(-(2 * y - 1) * (np.dot(Phi_z0, w) + b)))
        log_p_z0 = -np.log(1.0 + np.exp(-(2 * 0 - 1) * (np.dot(Phi_16, u) + a)))
        score_z0 = log_p_r_z0 + log_p_z0
        
        X_z1 = np.zeros((N, D + 1))
        X_z1[:, :8] = X[:, :8]
        X_z1[:, 8] = 1
        X_z1[:, 9:] = X[:, 8:]
        Phi_z1 = get_puf_embedding(X_z1)
        
        # Log-likelihood components for z = 1
        log_p_r_z1 = -np.log(1.0 + np.exp(-(2 * y - 1) * (np.dot(Phi_z1, w) + b)))
        log_p_z1 = -np.log(1.0 + np.exp(-(2 * 1 - 1) * (np.dot(Phi_16, u) + a)))
        score_z1 = log_p_r_z1 + log_p_z1
        
        z_new = np.where(score_z1 > score_z0, 1, 0)
        
        if np.array_equal(z, z_new):
            break
        z = z_new
        
    return w, b, u, a

In [ ]:
import numpy as np
import time
import os

# Try importing from your submission script
try:
    from submit import my_latent, my_latent_updated
except ImportError:
    print("Error: submit.py file not found in the same directory.")
    exit()

def simulate_mock_data():
    """Generates synthetic data matching the expected assignment structure."""
    print("Generating mock data for verification...")
    np.random.seed(42)
    N = 8000
    X = np.random.randint(0, 2, size=(N, 16))
    y = np.random.randint(0, 2, size=N)
    return X, y

if __name__ == "__main__":
    # Check for actual data files or fall back to simulated data
    if os.path.exists("X_train.npy") and os.path.exists("y_train.npy"):
        X = np.load("X_train.npy")
        y = np.load("y_train.npy")
        print(Loaded training data from directory. Shape: {X.shape})
    else:
        X, y = simulate_mock_data()
        
    # --- Test Trivial Prior Latent Optimization ---
    print("\n--- Testing Part 2: Trivial Prior EM ---")
    start_time = time.perf_counter()
    w, b, z = my_latent(X, y)
    elapsed = time.perf_counter() - start_time
    
    print(f"Execution successful! [Time taken: {elapsed:.2f} seconds]")
    print(f"w shape: {w.shape}, b scalar: {b:.4f}, z shape: {z.shape}")
    print(f"Recovered Latent Bit 1s Ratio: {np.mean(z):.4f}")
    
    # --- Test Structured Prior Latent Optimization ---
    print("\n--- Testing Part 4: Structured Prior EM ---")
    start_time = time.perf_counter()
    w_up, b_up, u, a = my_latent_updated(X, y)
    elapsed_up = time.perf_counter() - start_time
    
    print(f"Execution successful! [Time taken: {elapsed_up:.2f} seconds]")
    print(f"w_updated shape: {w_up.shape}, b_updated scalar: {b_up:.4f}")
    print(f"u shape: {u.shape}, a scalar: {a:.4f}")
    
    # Measure congruence as requested in Part 5
    cos_sim = np.dot(w, w_up) / (np.linalg.norm(w) * np.linalg.norm(w_up))
    print(f"\nCosine Similarity between w and w_updated: {cos_sim:.4f}")